# Atlas World — one-cell Kaggle build

Enable Internet and add the Kaggle Secret `GITHUB_TOKEN` before running the next cell. The secret must be a fine-grained GitHub token with Contents read/write access only to `crestog/unemployed-nigg-`. The cell clones the latest code, downloads the source-backed worldwide administrative and place data, builds and validates static MVT assets, archives them, and pushes only a complete release.

In [ ]:
import os
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path
from kaggle_secrets import UserSecretsClient

repo = Path('/kaggle/working/atlas-repo')
if repo.exists():
    shutil.rmtree(repo)

try:
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
except Exception as exc:
    raise RuntimeError('Add Kaggle Secret GITHUB_TOKEN with Contents read/write access to crestog/unemployed-nigg-.') from exc
if not token:
    raise RuntimeError('GITHUB_TOKEN is empty.')
os.environ['GITHUB_TOKEN'] = token
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['ATLAS_BUILD_LOG'] = '/kaggle/working/atlas-build.log'
release = 'world-global-geoboundaries-kaggle-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
workers = max(1, min(4, os.cpu_count() or 1))

def run(command, cwd=None):
    print('\n$ ' + ' '.join(map(str, command)), flush=True)
    subprocess.run(command, cwd=cwd, env=os.environ.copy(), check=True)

print('1/5 Clone latest code', flush=True)
run(['git', 'clone', '--depth', '1', '--branch', 'main', 'https://github.com/crestog/unemployed-nigg-', str(repo)])
run(['git', '-C', str(repo), 'log', '-1', '--oneline'])

print('2/5 Install build dependencies in the Kaggle kernel', flush=True)
run([sys.executable, '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check', 'ijson', 'shapely', 'antimeridian', 'mapbox-vector-tile', 'pyclipper', 'protobuf<6', 'pyproj', 'numpy'])

print('3/5 Hardware check: the geometry/MVT path is CPU-native', flush=True)
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', gpu.stdout.strip() if gpu.returncode == 0 else 'not available', flush=True)
print('CPU workers:', workers, flush=True)

print('4/5 Download, prepare, build, audit, validate, archive, commit, push', flush=True)
try:
    run([sys.executable, str(repo / 'scripts' / 'kaggle_build_atlas_world.py'), '--repo-dir', str(repo), '--repo-url', 'https://github.com/crestog/unemployed-nigg-.git', '--branch', 'main', '--release-id', release, '--archive', f'/kaggle/working/atlas-world-{release}.tar.gz', '--skip-install', '--workers', str(workers)], cwd=repo)
except subprocess.CalledProcessError:
    log = Path('/kaggle/working/atlas-build.log')
    print('\nBUILD FAILED. Last output from the child process:', flush=True)
    if log.exists():
        print(''.join(log.read_text(errors='replace').splitlines(True)[-100:]), flush=True)
    raise

print('5/5 SUCCESS: validated worldwide release pushed automatically', flush=True)
print('Release:', release, flush=True)
print('Actions: https://github.com/crestog/unemployed-nigg-/actions', flush=True)
print('Live site: https://unemployed-nigg.sahudevansh482.workers.dev/#world=1', flush=True)